In [1]:
%load_ext sql
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
# NOTE: the connection name (URL) is required as an argument.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

In [4]:
%%sql
CREATE SCHEMA IF NOT EXISTS bronze;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


# 1. Add Zenodo dataset into DuckDB

In [4]:
import csv
import re

# DuckDB 1.1.3's read_csv has NO 'encoding' parameter (unlike older versions),
# so the cp1252 file must be converted to UTF-8 first. The raw file also contains
# control characters and unescaped double-quotes that break DuckDB's strict CSV
# parser, so we re-emit a properly-quoted UTF-8 CSV with Python's csv module.

SRC = '/workspace/data/RecipeZenodo.csv'
TMP = '/workspace/data/RecipeZenodo_utf8.csv'
OUT = '/workspace/data/RecipeZenodo_final.csv'

# Step 1: cp1252 -> UTF-8 and strip control characters (e.g. NUL)
CTRL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
with open(SRC, 'rb') as fh, open(TMP, 'w', newline='', encoding='utf-8') as fo:
    for line in fh:
        fo.write(CTRL.sub('', line.decode('cp1252')))

# Step 2: re-quote fields properly (fixes unescaped double-quotes)
with open(TMP, newline='', encoding='utf-8') as fh, open(OUT, 'w', newline='', encoding='utf-8') as fo:
    reader = csv.reader(fh, delimiter=';', quotechar='"')
    writer = csv.writer(fo, delimiter=';', quotechar='"', quoting=csv.QUOTE_MINIMAL, lineterminator='\n')
    for row in reader:
        if len(row) != 18:  # skip malformed rows
            continue
        writer.writerow(row)

print('Preprocessed CSV ready:', OUT)

Preprocessed CSV ready: /workspace/data/RecipeZenodo_final.csv


In [5]:
%%sql
CREATE OR REPLACE TABLE bronze.RecipeZenodo AS
SELECT * FROM read_csv(
    '/workspace/data/RecipeZenodo_final.csv',
    delim=';',
    header=true,
    quote='"',
    ignore_errors=true,
    null_padding=true,
    parallel=false
);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [6]:
%%sql
SELECT COUNT(*) AS row_count FROM bronze.RecipeZenodo;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

row_count
50511


# 2. Add FoodDB and FoodUSDA datasets into DuckDB

In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS FooDB;
CREATE SCHEMA IF NOT EXISTS FoodUSDA_latest;
CREATE SCHEMA IF NOT EXISTS FoodUSDA_legacy;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [ ]:
%%sql
CREATE OR REPLACE TABLE FooDB.flavor AS
SELECT * FROM read_csv('/workspace/data/foodb_2020_04_07/Flavor.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [7]:
%%sql
CREATE OR REPLACE TABLE FooDB.food AS
SELECT * FROM read_csv('/workspace/data/foodb_2020_04_07/Food.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [8]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_latest.food AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_latest/food.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [9]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_latest.nutrient AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_latest/nutrient.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [10]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_latest.food_nutrient AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_latest/food_nutrient.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [11]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_legacy.food AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_legacy/food.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [12]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_legacy.nutrient AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_legacy/nutrient.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [13]:
%%sql
CREATE OR REPLACE TABLE FoodUSDA_legacy.food_nutrient AS
SELECT * FROM read_csv('/workspace/data/FoodUSDA_legacy/food_nutrient.csv', header=true, sample_size=-1);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [14]:
%%sql
SELECT 'FooDB.flavor' AS table_name, COUNT(*) AS row_count FROM FooDB.flavor
UNION ALL SELECT 'FooDB.food', COUNT(*) FROM FooDB.food
UNION ALL SELECT 'FoodUSDA_latest.food', COUNT(*) FROM FoodUSDA_latest.food
UNION ALL SELECT 'FoodUSDA_latest.nutrient', COUNT(*) FROM FoodUSDA_latest.nutrient
UNION ALL SELECT 'FoodUSDA_latest.food_nutrient', COUNT(*) FROM FoodUSDA_latest.food_nutrient
UNION ALL SELECT 'FoodUSDA_legacy.food', COUNT(*) FROM FoodUSDA_legacy.food
UNION ALL SELECT 'FoodUSDA_legacy.nutrient', COUNT(*) FROM FoodUSDA_legacy.nutrient
UNION ALL SELECT 'FoodUSDA_legacy.food_nutrient', COUNT(*) FROM FoodUSDA_legacy.food_nutrient;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

table_name,row_count
FooDB.flavor,883
FooDB.food,992
FoodUSDA_latest.food,87990
FoodUSDA_latest.nutrient,477
FoodUSDA_latest.food_nutrient,170469
FoodUSDA_legacy.food,7793
FoodUSDA_legacy.nutrient,474
FoodUSDA_legacy.food_nutrient,644125
